# Trading Strategy DSL - Complete Guide

**Location:** `D:\Projects_Main\Finacal_Adviser\nlp-to-strategy-engine\dsl`

This notebook demonstrates the complete DSL (Domain-Specific Language) system for trading strategies.

## 🔄 What Transformer does

Transformer walks the parse tree and replaces grammar rules with Python objects.

DSL text

 → Grammar matches

 → Parse tree created

 → Transformer walks tree (bottom-up)
 
 → AST nodes created


## 1. Setup and Imports

In [2]:
import sys
import os

# Add parent directory to path
sys.path.insert(0, r'D:\Projects_Main\Finacal_Adviser\nlp-to-strategy-engine')

from dsl import parse_dsl, validate_dsl, get_strategy_quality
from dsl.ast_nodes import Strategy, Comparison, BooleanOp
from typing import List
from pydantic import BaseModel

print("✅ Imports successful!")

✅ Imports successful!


## 2. What is DSL?

A **Domain-Specific Language** that converts human-readable trading rules into structured code.

### Example:
```
Human: "Buy when RSI crosses 70, sell when it drops below 30"
↓
DSL: ENTRY: rsi(close, 14) > 70
     EXIT: rsi(close, 14) < 30
↓
Code: Executable strategy
```

## 3. DSL Syntax Overview

### Basic Structure
```
ENTRY: <condition>
EXIT: <condition>
```

### Operators
- **Comparison:** `>`, `<`, `>=`, `<=`, `==`, `!=`
- **Cross:** `crosses_above`, `crosses_below`, `crosses`, `touches`
- **Boolean:** `AND`, `OR`, `(parentheses)`

### Data Types
- **Series:** `close`, `open`, `high`, `low`, `volume`
- **Numbers:** `100`, `1.5K`, `2M`, `1B`
- **Time refs:** `close_prev`, `high_5d_ago`

### Indicators
- `sma(close, 20)` - Simple Moving Average
- `ema(close, 12)` - Exponential Moving Average
- `rsi(close, 14)` - Relative Strength Index
- `macd(close, 12, 26, 9)` - MACD
- `bb_upper(close, 20, 2)` - Bollinger Band Upper
- And more...

## 4. Example 1: Simple RSI Strategy

In [3]:
# Simple RSI strategy
dsl_text = """
ENTRY: rsi(close, 14) > 70
EXIT: rsi(close, 14) < 30
"""

print("📝 DSL Text:")
print(dsl_text)

# Parse the DSL
ast = parse_dsl(dsl_text)
print("\n✅ Parsed AST:")
print(f"Type: {type(ast).__name__}")
print(f"Has Entry: {ast.entry is not None}")
print(f"Has Exit: {ast.exit is not None}")

# Validate
is_valid, errors, warnings = validate_dsl(ast)
print(f"\n✅ Validation: {'PASSED' if is_valid else 'FAILED'}")
if errors:
    print(f"Errors: {errors}")
if warnings:
    print(f"Warnings: {warnings}")

# Quality check
quality = get_strategy_quality(ast)
print("\n📊 Quality Metrics:")
print(f"Entry Complexity: {quality['entry_complexity']}")
print(f"Exit Complexity: {quality['exit_complexity']}")
print(f"Indicator Count: {quality['indicator_count']}")
print(f"Has Exit: {quality['has_exit']}")
if quality['warnings']:
    print(f"Warnings: {quality['warnings']}")

📝 DSL Text:

ENTRY: rsi(close, 14) > 70
EXIT: rsi(close, 14) < 30


✅ Parsed AST:
Type: Strategy
Has Entry: True
Has Exit: True

✅ Validation: PASSED

📊 Quality Metrics:
Entry Complexity: 1
Exit Complexity: 1
Indicator Count: 1
Has Exit: True


## 5. Example 2: Complex Multi-Indicator Strategy

In [4]:
# Complex strategy with multiple indicators
dsl_text = """
ENTRY: sma(close, 20) > sma(close, 50) AND rsi(close, 14) < 40 AND volume > 1M
EXIT: rsi(close, 14) > 70 OR close < sma(close, 20)
"""

print("📝 DSL Text:")
print(dsl_text)

ast = parse_dsl(dsl_text)
print("\n✅ Parsed Successfully!")

is_valid, errors, warnings = validate_dsl(ast)
print(f"✅ Valid: {is_valid}")

quality = get_strategy_quality(ast)
print("\n📊 Quality Metrics:")
print(f"Entry Complexity: {quality['entry_complexity']}")
print(f"Exit Complexity: {quality['exit_complexity']}")
print(f"Total Indicators: {quality['indicator_count'] + quality['exit_indicator_count']}")
if quality['warnings']:
    print(f"⚠️  Warnings: {quality['warnings']}")

📝 DSL Text:

ENTRY: sma(close, 20) > sma(close, 50) AND rsi(close, 14) < 40 AND volume > 1M
EXIT: rsi(close, 14) > 70 OR close < sma(close, 20)


✅ Parsed Successfully!
✅ Valid: True

📊 Quality Metrics:
Entry Complexity: 3
Exit Complexity: 2
Total Indicators: 5


## 6. Example 3: Cross Event Strategy (Golden Cross)

In [5]:
# Golden Cross / Death Cross strategy
dsl_text = """
ENTRY: sma(close, 20) crosses_above sma(close, 50)
EXIT: sma(close, 20) crosses_below sma(close, 50)
"""

print("📝 DSL Text (Golden Cross):")
print(dsl_text)

ast = parse_dsl(dsl_text)
print("\n✅ Parsed Successfully!")

is_valid, errors, warnings = validate_dsl(ast)
print(f"✅ Valid: {is_valid}")

quality = get_strategy_quality(ast)
print("\n📊 Quality:")
print(f"Complexity: {quality['entry_complexity']} (Simple)")
print(f"Indicators: {quality['indicator_count']}")

📝 DSL Text (Golden Cross):

ENTRY: sma(close, 20) crosses_above sma(close, 50)
EXIT: sma(close, 20) crosses_below sma(close, 50)


✅ Parsed Successfully!
✅ Valid: True

📊 Quality:
Complexity: 1 (Simple)
Indicators: 2


## 7. Example 4: Bollinger Bands Strategy

In [6]:
# Bollinger Bands mean reversion
dsl_text = """
ENTRY: close < bb_lower(close, 20, 2)
EXIT: close > bb_upper(close, 20, 2)
"""

print("📝 DSL Text (Bollinger Bands):")
print(dsl_text)

ast = parse_dsl(dsl_text)
is_valid, errors, warnings = validate_dsl(ast)

print(f"\n✅ Valid: {is_valid}")
print(f"Entry: {ast.entry}")
print(f"Exit: {ast.exit}")

📝 DSL Text (Bollinger Bands):

ENTRY: close < bb_lower(close, 20, 2)
EXIT: close > bb_upper(close, 20, 2)


✅ Valid: True
Entry: Comparison(node_type='comparison', operator='<', left=Series(node_type='series', name='close'), right=Indicator(node_type='indicator', name='bb_lower', params=['close', 20, 2]))
Exit: Comparison(node_type='comparison', operator='>', left=Series(node_type='series', name='close'), right=Indicator(node_type='indicator', name='bb_upper', params=['close', 20, 2]))


## 8. Example 5: Using Scaled Numbers

In [7]:
# Strategy with K/M/B scaled numbers
dsl_text = """
ENTRY: volume > 1.5M AND close > 100
EXIT: volume < 500K OR close < 95
"""

print("📝 DSL Text (Scaled Numbers):")
print(dsl_text)
print("\nNote: 1.5M = 1,500,000 | 500K = 500,000")

ast = parse_dsl(dsl_text)
is_valid, errors, warnings = validate_dsl(ast)

print(f"\n✅ Valid: {is_valid}")

📝 DSL Text (Scaled Numbers):

ENTRY: volume > 1.5M AND close > 100
EXIT: volume < 500K OR close < 95


Note: 1.5M = 1,500,000 | 500K = 500,000

✅ Valid: True


## 9. Converting NLP Output to DSL

This shows how to convert ParsedStrategy (from NLP) to DSL format.

In [8]:
# Mock NLP Output Models
class Condition(BaseModel):
    left: str
    operator: str
    right: float | str

class TradingRule(BaseModel):
    entry: List[Condition]
    exit: List[Condition]
    initial_capital: float
    position_size: float

class ParsedStrategy(BaseModel):
    rule: TradingRule
    original_text: str
    indicators_used: List[str]
    complexity: str
    initial_capital: float
    position_size: float

# Converter function
def convert_to_dsl(parsed_strategy: ParsedStrategy) -> str:
    """Convert ParsedStrategy to DSL text"""
    def condition_to_dsl(condition: Condition) -> str:
        right_str = str(condition.right) if not isinstance(condition.right, str) else condition.right
        return f"{condition.left} {condition.operator} {right_str}"
    
    entry_dsl = " AND ".join([condition_to_dsl(c) for c in parsed_strategy.rule.entry])
    exit_dsl = " AND ".join([condition_to_dsl(c) for c in parsed_strategy.rule.exit])
    
    dsl_text = f"ENTRY: {entry_dsl}"
    if exit_dsl:
        dsl_text += f"\nEXIT: {exit_dsl}"
    return dsl_text

print("✅ Converter function ready!")

✅ Converter function ready!


## 10. Complete NLP → DSL → Parse Example

In [9]:
# Mock NLP output
parsed_strategy = ParsedStrategy(
    rule=TradingRule(
        entry=[Condition(left="rsi(close, 14)", operator=">", right=70)],
        exit=[Condition(left="rsi(close, 14)", operator="<", right=30)],
        initial_capital=50000.0,
        position_size=0.5
    ),
    original_text="Buy when RSI > 70 with $50k invest 50%. Sell when RSI < 30.",
    indicators_used=["rsi"],
    complexity="simple",
    initial_capital=50000.0,
    position_size=0.5
)

print("📝 Original Text:")
print(parsed_strategy.original_text)
print(f"\n💰 Capital: ${parsed_strategy.initial_capital:,.0f}")
print(f"📊 Position: {parsed_strategy.position_size * 100}%")

# Convert to DSL
dsl_text = convert_to_dsl(parsed_strategy)
print("\n🔧 Generated DSL:")
print(dsl_text)

# Parse DSL
ast = parse_dsl(dsl_text)
print("\n✅ Parsed AST:")
print(f"Type: {type(ast).__name__}")

# Validate
is_valid, errors, warnings = validate_dsl(ast)
print(f"\n✅ Validation: {'PASSED' if is_valid else 'FAILED'}")

# Quality
quality = get_strategy_quality(ast)
print("\n📊 Quality Metrics:")
for key, value in quality.items():
    if key != 'warnings':
        print(f"   {key}: {value}")
if quality['warnings']:
    print(f"   ⚠️  warnings: {quality['warnings']}")

📝 Original Text:
Buy when RSI > 70 with $50k invest 50%. Sell when RSI < 30.

💰 Capital: $50,000
📊 Position: 50.0%

🔧 Generated DSL:
ENTRY: rsi(close, 14) > 70.0
EXIT: rsi(close, 14) < 30.0

✅ Parsed AST:
Type: Strategy

✅ Validation: PASSED

📊 Quality Metrics:
   entry_complexity: 1
   exit_complexity: 1
   has_exit: True
   indicator_count: 1
   exit_indicator_count: 1


## 11. Error Handling Examples

In [10]:
# Example 1: Invalid indicator
print("Testing invalid indicator...")
try:
    dsl_text = "ENTRY: invalid_indicator(close, 20) > 100"
    ast = parse_dsl(dsl_text)
except Exception as e:
    print(f"❌ Error (expected): {e}")

print("\n" + "="*50)

# Example 2: Invalid operator
print("\nTesting invalid operator...")
try:
    dsl_text = "ENTRY: close >> 100"  # Invalid operator
    ast = parse_dsl(dsl_text)
except Exception as e:
    print(f"❌ Error (expected): {e}")

print("\n" + "="*50)

# Example 3: Missing ENTRY
print("\nTesting missing ENTRY...")
try:
    dsl_text = "EXIT: close < 100"  # No ENTRY
    ast = parse_dsl(dsl_text)
except Exception as e:
    print(f"❌ Error (expected): {e}")

Testing invalid indicator...
❌ Error (expected): DSL Parse Error: No terminal matches 'i' in the current parser context, at line 1 col 8

ENTRY: invalid_indicator(close, 20) > 100
       ^
Expected one of: 
	* LPAR
	* NUMBER_SCALED
	* SERIES_LITERAL
	* INDICATOR_NAME

Previous tokens: Token('COLON', ':')



Testing invalid operator...
❌ Error (expected): DSL Parse Error: Unexpected token Token('GT', '>') at line 1, column 15.
Expected one of: 
	* LPAR
	* NUMBER_SCALED
	* SERIES_LITERAL
	* INDICATOR_NAME
Previous tokens: [Token('GT', '>')]



Testing missing ENTRY...
❌ Error (expected): DSL Parse Error: Unexpected token Token('EXIT', 'EXIT') at line 1, column 1.
Expected one of: 
	* ENTRY
Previous tokens: [None]



## 12. DSL Module Structure

```
dsl/
├── grammar.lark          ← Defines syntax rules
├── ast_nodes.py          ← Data structures (Strategy, Comparison, etc.)
├── parser.py             ← Text → AST converter
├── validator.py          ← AST validator & quality checker
└── __init__.py           ← Exports all public functions
```

### Flow:
```
DSL Text → Parser → AST → Validator → Quality Check → Code Generation
```

## 13. Summary

### What We Covered:
1. ✅ DSL syntax and structure
2. ✅ Parsing DSL text to AST
3. ✅ Validation and quality checks
4. ✅ Converting NLP output to DSL
5. ✅ Error handling
6. ✅ Multiple strategy examples

### Key Functions:
- `parse_dsl(text)` - Parse DSL text to AST
- `validate_dsl(ast)` - Validate AST structure
- `get_strategy_quality(ast)` - Get quality metrics

### Next Steps:
- Use this DSL in your NLP pipeline
- Generate trading code from validated AST
- Backtest strategies
- Deploy to production

## 14. Try Your Own Strategy!

Modify the cell below to test your own DSL strategy:

In [11]:
# Your custom strategy here!
my_dsl_text = """
ENTRY: close > sma(close, 20) AND volume > 1M
EXIT: close < sma(close, 20)
"""

print("📝 Your DSL Strategy:")
print(my_dsl_text)

try:
    ast = parse_dsl(my_dsl_text)
    is_valid, errors, warnings = validate_dsl(ast)
    quality = get_strategy_quality(ast)
    
    print(f"\n✅ Valid: {is_valid}")
    print(f"📊 Complexity: {quality['entry_complexity']}")
    print(f"📈 Indicators: {quality['indicator_count']}")
    
    if quality['warnings']:
        print(f"⚠️  Warnings: {quality['warnings']}")
        
except Exception as e:
    print(f"\n❌ Error: {e}")

📝 Your DSL Strategy:

ENTRY: close > sma(close, 20) AND volume > 1M
EXIT: close < sma(close, 20)


✅ Valid: True
📊 Complexity: 2
📈 Indicators: 1
